<a href="https://colab.research.google.com/github/GiulianoPepato/Machine-learning-to-predict-cystal-structure-/blob/main/ML_crystal_structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Ambiente e Bibliotecas

In [ ]:
!pip3 install pandas mp_api mendeleev matminer pymatgen pymatviz

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pymatviz as pmv

from pymatviz import ptable_heatmap_plotly
from mp_api.client import MPRester
from collections import Counter
from collections import defaultdict
from pymatgen.core import Element, Structure, Composition
from mendeleev import element
from matminer.featurizers.composition import ElementProperty

from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix

from tensorflow import keras
from keras import layers

seed = 42
np.random.seed(seed)
random.seed(seed)

#Dowload dos dados

Para ser extremamente objetivo, precisamos das informações de elementos, fórmula, energia acima do hull, estrutura e simetria dos materiais cujas propriedades foram calculadas pelos funcionais GGA e GGA+U, porém a função que efetua essa busca `thermo_search` não busca uma das propriedades solicitadas e não consegue baixar uma grande quantidade de uma vez, portanto precisamos contruir um sistema de busca menos simples.

Primeiro é baixado os id's de todos os materiais binários e ternários do *Materials Project*, com essa lista é efetuado o dowload das informações de fórmula, id, energia acima do hull, elementos, simetria e energia de formação por átomo dos materiais no passo de mil em mil. Em seguida todos esses dados são filtrados de duas formas: eliminando todos os materiais com energia de formação por átomo maior do que 0 e, dado um conjunto de materiais que possuem os mesmos elementos de formação, é mantido apenas aquele que possui a menor energia acima do hull.

Dessa forma, é baixada as informações que eram necessárias a priori dos materiais que passaram pela filtragem e então é separado o conjunto de dados nos conjuntos de treinamento `df_train` e de validação `df_val` via Label Enconder apenas por praticidade.


In [ ]:
API_KEY = input('your MP API key: ')

with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        nelements= (2,3), fields = ['material_id']
    ) #dowload dos id's de materiais binários e ternários

mpids = [doc.material_id for doc in docs]

fields = ['formula_pretty',
          'material_id',
          'energy_above_hull',
          'elements',
          'symmetry',
          'formation_energy_per_atom',
          ]


with MPRester(API_KEY) as mpr:
    # thermo_docs = mpr.materials.thermo.search(material_ids=df['mpid'])
  i = 0
  Thermo_docs =[]
  # Use a smaller, safer chunk size for API calls to avoid rate limiting
  # and loop up to the actual length of mpids.
  # A chunk size of 500-1000 is often more stable for mp-api.
  small_chunk_size = 1000
  total_mpids = len(mpids) # Use actual length of mpids

  while i < total_mpids:
    # In addition, you can specify the level of theory by using "thermo_type", the default is "GGA_GGA+U_R2SCAN":
    thermo_docs = mpr.materials.thermo.search(
        material_ids=mpids[i:small_chunk_size+i], thermo_types=["GGA_GGA+U"], fields = fields)
    Thermo_docs += thermo_docs
    i += small_chunk_size # Increment by the new small_chunk_size

data = []
for entry in Thermo_docs:
  data.append(entry.model_dump())


df = pd.DataFrame(data)
df = df.drop(columns=[ 'fields_not_requested'])

df_0 = df[ ~( df['formation_energy_per_atom'] > 0)].reset_index(drop = True)
#excluíndo os dados cuja energia de formação por átomo é maior que 0

df_0['elements_tuple'] = df_0['elements'].apply(lambda x: tuple(sorted([el for el in x])))
tuple_elements = df_0['elements_tuple'].unique().tolist()
#lista de todas as tuplas (sem repetições) dos elementos dos materiais

indices_to_remove = [] #índices a remover
for elementos in tuple_elements: #para cada tupla na nossa lista
  indices = df_0.index[df_0['elements_tuple'] == elementos].tolist() #índices onde se encontram essa tupla
  ehull = [] #lista de energia acima do hull dos materiais nesses índices
  if len(indices) > 1: #se haver apenas um índice/material com essa tupla, ele passa para o próximo
    for ind in indices:
      ehull.append(df_0['energy_above_hull'][ind]) #adiciona o valor da energia acima do hull de cada um dos materiais
    indices.pop(ehull.index(min(ehull))) #lista dos índices, excluíndo o que tem menor energia acima do hull
    indices_to_remove += indices #justa em uma lista todos os índices que devem ser removidos
  else:
    pass

df_1 = df_0.drop(index = indices_to_remove).reset_index(drop = True) #dados filtrados

materials_id = df_1['material_id'].tolist()
fields = [ 'elements','structure', 'symmetry', 'formula_pretty', 'energy_above_hull']

with MPRester(API_KEY) as mpr:
  Docs = []
  i = 0
  divisor = 5029
  while i < 20116:
    if i == 3*divisor:
      docs = mpr.materials.summary.search(
        material_ids = materials_id[i:], fields = fields
    )
    else:
      docs = mpr.materials.summary.search(
          material_ids = materials_id[i:divisor+i], fields = fields
      )
    i += divisor
    Docs += docs

exemple_doc = Docs[0]
data = []
for entry in Docs:
  data.append(entry.model_dump())


DFm = pd.DataFrame(data)
DFm = DFm.drop(columns=  ['fields_not_requested'])
DFm['Crystal System'] = DFm['symmetry'].apply(lambda x: x.get('crystal_system'))


le = LabelEncoder()
le.fit(DFm['Crystal System'].astype(str))
y_le = list(le.transform( DFm['Crystal System'].astype(str) ))


X = DFm.drop(columns = ['symmetry'])
from sklearn.model_selection import train_test_split
df_train, df_val, y_train, y_val = train_test_split(X, y_le, test_size = 0.2, random_state = seed, stratify = y_le)

# Vizualição dos materiais

As funções abaixo servem para observar os materiais dos nossos conjuntos de diferentes formas: `periodic_table` gera uma tabela períodica com a quantidade de cada elemento no conjunto de dados que quisermos, `structure_distribuction` gera uma imagem com a distribuição das nossas estrutura cristalina no conjunto que quisermos e a função `strucuture_only` retorna um data frame apenas com a estrutura cristalina desejada.

In [ ]:
def periodic_table(dataframe):
  nome = input('Enter dataframe name: ')
  element_counts = {}

  for f in dataframe["formula"]:
      comp = Composition(f)
      for el in comp.elements:
          element_counts[el.symbol] = element_counts.get(el.symbol, 0) + 1

  fig = pmv.ptable_heatmap_plotly(element_counts, colorscale="YlGn")


  fig.update_layout(
      title=f"Element Occurrence in Stable Binary Compounds from {nome}<br>(Total: {len(dataframe['formula'])})",
      title_x=0.41,  # center the title
      title_y=0.9
  )

  # Show interactive plot
  fig.show()

def structure_distribuation(dataframe):
  nome = input('Enter data frame name: ')
  df_1 = dataframe.copy()

  # Define the mapping for renaming crystal systems
  crystal_system_mapping = {
      'Cubic': 'cub',
      'Orthorhombic': 'orth',
      'Hexagonal': 'hex',
      'Tetragonal': 'tetra',
      'Trigonal': 'trig',
      'Monoclinic': 'mono',
      'Triclinic': 'tric'
  }

  # Apply the renaming to the 'Crystal System' column
  df_1['Crystal System'] = df_1['Crystal System'].replace(crystal_system_mapping)

  data = df_1['Crystal System'].value_counts()

  plt.figure()
  plt.title(f'Distribution of crystal systems from {nome}')

  sns.barplot(x=data.index, y=data.values, )
  plt.show()

def structure_only(dataframe):
  structures = dataframe['Crystal System'].unique().tolist()
  print('Enter a number to select the structure:')
  for i,j in enumerate(structures):
    print(f'{i+1} - {j}')
  estrutura = int(input(""))
  return dataframe[ (dataframe['Crystal System'] == structures[estrutura-1])]

#Features do Mendeleev

A função `data_augmentation` serve para criar todas as permutações possíveis (6) da ordem dos elementos dos materiais e seus dados. `element_data` cria catálogos (dicionários) das informações dos elementos presentes em cada um dos materiais, contudo algumas informações estão faltantes na biblioteca *mendeleev*, por isso impomos os valores manualmente na célula seguinte. A função `ajuda_feature` está presente apenas para fazer o trabalho de colocar todas as features na função `featurization`.

Como o conjunto de validação serve apenas para ser avaliado, não existe sentido em querer permutar a ordem dos materiais.

In [ ]:
def data_augmantation(dataframe):

  dataframe['1 element'] = dataframe['elements'].apply(lambda x: str(x[0]))
  dataframe['2 element'] = dataframe['elements'].apply(lambda x: str(x[1]))
  dataframe['3 element'] = dataframe['elements'].apply(lambda x: str(x[2]) if len(x) == 3 else None)


  formulas = dataframe['formula_pretty'].tolist()
  pelemento = dataframe['1 element'].tolist()
  selemento = dataframe['2 element'].tolist()
  telemento = dataframe['3 element'].tolist()
  simetria =  dataframe['Crystal System'].tolist()
  estrutura = dataframe['structure'].tolist()
  ehull = dataframe['energy_above_hull'].tolist()

  return pd.DataFrame(data = {'1 element': pelemento+pelemento+selemento+selemento+telemento+telemento,
                              '2 element': selemento+telemento+telemento+pelemento+pelemento+selemento,
                              '3 element': telemento+selemento+pelemento+telemento+selemento+pelemento,
                              'Crystal System': simetria+simetria+simetria+simetria+simetria+simetria,
                              'structure': estrutura+estrutura+estrutura+estrutura+estrutura+estrutura,
                              'formula': formulas+formulas+formulas+formulas+formulas+formulas,
                              'energy_above_hull': ehull + ehull + ehull + ehull + ehull + ehull})

def element_data(lista_elementos): # This function creat dictionarys with atomic number, atomic radius, eletrons and electron affinity for wich element
  e_atoms = {}
  n_atoms = {}
  r_atoms = {}
  eletron_val = {}
  for elem in lista_elementos:
    if elem == None:
      pass
    else:
      elemento = element(str(elem))
      n_atoms[str(elem)] = elemento.atomic_number
      r_atoms[str(elem)] = elemento.atomic_radius
      e_atoms[str(elem)] = elemento.electron_affinity
      eletron_val[str(elem)] = elemento.ec.last_subshell()[1]
  return n_atoms, r_atoms, e_atoms, eletron_val

def ajuda_feature(catalogo, atributo, DFmaterial):
  for i in range(3):
    n = str(i+1)
    atributo_n = atributo + '_' + n
    elemento = n + " " + 'element'
    DFmaterial[atributo_n] = DFmaterial[elemento].apply(lambda x: 0 if x == None else catalogo[x])
  return DFmaterial

In [ ]:
df_train = data_augmantation(df_train)

df_val['1 element'] = df_val['elements'].apply(lambda x: str(x[0])) # Here I separete the column 'elements' in two others (the first element and the second)
df_val['2 element'] = df_val['elements'].apply(lambda x: str(x[1]))
df_val['3 element'] = df_val['elements'].apply(lambda x: str(x[2]) if len(x) == 3 else None)
df_val = df_val.drop(columns = 'elements').rename(columns = {'formula_pretty' : 'formula'})

df_total = pd.concat([df_train, df_val]) #I need to creat a data frame with all elements to creat a list with them
elementos = list(set(df_total['1 element']))

cat_natoms, cat_ratoms, cat_eatoms, cat_eletron_v = element_data(elementos)

cat_ratoms['Kr'] = 88.0 # In mendleev data base this elements don't have their values, but Wikipédia have it.
cat_ratoms['Xe'] = 108.0

cat_eatoms['Mg'] = -0.4
cat_eatoms['Kr'] = -1.0
cat_eatoms['Pm'] = 0.129
cat_eatoms['Ho'] = 0.338
cat_eatoms['Pa'] = 0.55
cat_eatoms['Cd'] = -0.7
cat_eatoms['Pu'] = -0.50
cat_eatoms['Th'] = 0.60769
cat_eatoms['Zn'] = -0.6
cat_eatoms['Hg'] = -0.5
cat_eatoms['Er'] = 0.312
cat_eatoms['Mn'] = -0.5
cat_eatoms['Np'] = 0.48
cat_eatoms['U'] = 0.31497
cat_eatoms['Sm'] = 0.162
cat_eatoms['Gd'] = 0.212

def featurization(DFmaterial, cat_natoms = cat_natoms, cat_ratoms = cat_ratoms, cat_eatoms = cat_eatoms, cat_eletron_v = cat_eletron_v):

  DFmaterial = ajuda_feature(cat_natoms, 'Z', DFmaterial)
  DFmaterial = ajuda_feature(cat_ratoms, 'radius', DFmaterial)
  DFmaterial = ajuda_feature(cat_eletron_v, 'e_val', DFmaterial)
  DFmaterial = ajuda_feature(cat_eatoms, 'e_affny', DFmaterial)

  return DFmaterial

df_train = featurization(df_train)
df_val = featurization(df_val)

#Features pelo Pymatgen

O conjunto de funções abaixo é feito apenas para criar as features das quantidades estequiométricas de cada elemento dos materiais.

In [ ]:
def count_elements_in_sites(structure_dict):
  """
  Counts the occurrences of each element in a structure dictionary.

  Args:
    structure_dict: A dictionary representing the structure.

  Returns:
    A dictionary where keys are element symbols and values are their counts.
  """
  # Create a Structure object from the dictionary
  structure = Structure.from_dict(structure_dict)
  element_counts = Counter(site.species_string for site in structure.sites)
  return dict(element_counts)

def estq_n(structure_dict, elemento):
  """
  Gets the stoichiometric quantity of a specific element from a structure dictionary.

  Args:
    structure_dict: A dictionary representing the structure.
    elemento: The element symbol (string).

  Returns:
    The count of the element in the structure, or 0 if not present.
  """
  estq = count_elements_in_sites(structure_dict)
  return estq.get(elemento, 0)

def estequiometrizacao(dataframe):

  dataframe['qnt 1'] = dataframe.apply(lambda row: estq_n(row['structure'], row['1 element']), axis = 1)
  dataframe['qnt 2'] = dataframe.apply(lambda row: estq_n(row['structure'], row['2 element']), axis = 1)
  dataframe['qnt 3'] = dataframe.apply(lambda row: estq_n(row['structure'], row['3 element']), axis = 1)

  return dataframe.drop(columns = ['structure']).rename(columns = {'formula_pretty': 'formula'})

In [ ]:
df_train = estequiometrizacao(df_train)
df_val = estequiometrizacao(df_val)

#Features do Matminer

A função `magpie_features` atribui a features de eletromegatividade do nosso material baseado nos seus elementos químicos. No passo seguinte é feito um data augmentation dos materiais triclinicos do conjunto de treinamento, criando duas novas "cópias" das entradas desses materiais, mas com uma variação gaussiana dos valores da eletronegatividade com a incerteza sendo o desvio padrão da média (`avg_dev`) dessa grandeza.

In [ ]:
def magpie_features(dataframe):

  stat = ["mean"]

  # Convert string → Composition
  dataframe['composition'] = dataframe["formula"].apply(Composition)
  compositions = dataframe["composition"].unique().tolist()

  # Select properties and statistics
  properties = ["Electronegativity"]

  # Initialize featurizer
  ep_stat = ElementProperty(data_source="magpie", features=properties, stats=stat)

  values_eletronegativity = {}
  values_meltingT = {}
  for comp in compositions:
    values_eletronegativity[str(comp)] = ep_stat.featurize(comp)




  dataframe['Electronegativity'] = dataframe['composition'].apply(lambda x: values_eletronegativity[str(x)][0])
  return dataframe

In [ ]:
df_train = magpie_features(df_train)
df_val = magpie_features(df_val)

triclinic_system = 'Triclinic'
df_tric = df_train[df_train['Crystal System'] == triclinic_system].copy()

ep_stat = ElementProperty(data_source="magpie", features=['Electronegativity'], stats=['avg_dev'])
df_tric = ep_stat.featurize_dataframe(df_tric, col_id='composition', ignore_errors=True)
df_tric = df_tric.rename(columns = {'MagpieData avg_dev Electronegativity': 'avg_dev'})

n_augmented_copies = 2
augmented_data_frames = []
for _ in range(n_augmented_copies):
    tric_copy = df_tric.copy()
    tric_copy['Electronegativity'] = tric_copy.apply(
        lambda row: row['Electronegativity'] * np.random.randn() * row['avg_dev'],
        axis=1
    )
    augmented_data_frames.append(tric_copy)

df_tric = pd.concat([df_tric] + augmented_data_frames, ignore_index=True)

df_train = df_train[~(df_train['Crystal System'] == 'Triclinic')]
df_train = pd.concat([df_train, df_tric], ignore_index = True).drop(columns = ['avg_dev'])

In [ ]:
df_train.columns

In [ ]:
print('Missing values in df_train["Electronegativity"]:')
missing_train = df_train[df_train['Electronegativity'].isna()]
if not missing_train.empty:
    print(missing_train['formula'].tolist())
else:
    print('No missing values found in df_train.')

print('\nMissing values in df_val["Electronegativity"]:')
missing_val = df_val[df_val['Electronegativity'].isna()]
if not missing_val.empty:
    print(missing_val['formula'].tolist())
else:
    print('No missing values found in df_val.')

print('Missing values in df_train["Melting T"]:')
missing_train = df_train[df_train['Melting T'].isna()]
if not missing_train.empty:
    print(missing_train['formula'].tolist())
else:
    print('No missing values found in df_train.')

print('\nMissing values in df_val["Melting T"]:')
missing_val = df_val[df_val['Melting T'].isna()]
if not missing_val.empty:
    print(missing_val['formula'].tolist())
else:
    print('No missing values found in df_val.')

#Aplicação do Machine Learning

As funções na célula seguinte seguir servem apenas para facilitar na hora de avaliar o desempenho dos algoritmos de aprendizado de máquina. É excluído as colunas de quais são os elementos dos conjuntos de treinamento e validação porque não nos interessam mais, transformamos os dados categoricos do nome das estruturas em dados númericos pelo Label Enconder, nos conjuntos `X` e `X_val` passamos as features que o modelo usará para treinar, escalolamos os dados e então avaliamos os modelos. Além disso o conjunto de treinamento é novamente divido entre o definitivo conjunto de treinamento e o conjunto de teste que servirá para avaliar se há overfitting dos dados.

In [ ]:
def c_matrix(y, y_pred, nome):
  # Calculate the confusion matrix
  cm = confusion_matrix(y, y_pred)

  # Normalize the confusion matrix to show percentages
  cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

  # Get the class labels from the LabelEncoder
  class_labels = le.classes_

  # Plot the confusion matrix
  plt.figure(figsize=(7, 5.5))
  # sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
  sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.title(f'Confusion Matrix for validation set from {nome} model')
  plt.show()

def resultados(modelo, y_test,y_pred_test, y_val, y_pred_val):
  print(modelo)
  print('')
  print('Métricas no conjunto de teste')
  print('Accuracy = ', accuracy_score(y_test, y_pred_test))
  print('Precision = ', precision_score(y_test, y_pred_test, average = 'macro'))
  print('F1 = ', f1_score(y_test, y_pred_test, average = 'macro'))
  print('Recall = ', recall_score(y_test, y_pred_test, average = 'macro'))
  print('')
  print('')
  print('Métricas no conjunto de validação')
  print('Accuracy = ', accuracy_score(y_val, y_pred_val))
  print('Precision = ', precision_score(y_val, y_pred_val, average = 'macro'))
  print('F1 = ', f1_score(y_val, y_pred_val, average = 'macro'))
  print('Recall = ', recall_score(y_val, y_pred_val, average = 'macro'))
  print('')
  c_matrix(y_val, y_pred_val, modelo)

In [ ]:
colunas = ['1 element', '2 element', '3 element']
df_train = df_train.drop(columns = colunas)
df_val = df_val.drop(columns = colunas)

le = LabelEncoder()
le.fit(df_train['Crystal System'].astype(str))
y_le = list(le.transform( df_train['Crystal System'].astype(str) ))
y_val = list(le.transform( df_val['Crystal System'].astype(str) ))

colunas = ['Crystal System', 'formula', 'composition', 'energy_above_hull']
X = df_train.drop(columns = colunas)
X_val = df_val.drop(columns = colunas)
X_train, X_test, y_train, y_test = train_test_split(X, y_le, test_size=0.1, random_state=seed, stratify = y_le)

scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
X_val_s =  scaler.transform(X_val)

## Modelos Ensemble

In [ ]:
rf = RandomForestClassifier(random_state = seed)
rf.fit(X_train_s, y_train)
y_pred_test = rf.predict(X_test_s)
y_pred_val = rf.predict(X_val_s)

resultados(modelo = 'Random Forest', y_pred_test=y_pred_test, y_test=y_test,
           y_val = y_val, y_pred_val = y_pred_val)

In [ ]:
gb = GradientBoostingClassifier(random_state=seed)
gb.fit(X_train_s, y_train)
y_pred_test = gb.predict(X_test_s)
y_pred_val = gb.predict(X_val_s)

resultados(modelo = 'Gradient Boost', y_pred_test=y_pred_test, y_test=y_test,
           y_val = y_val, y_pred_val = y_pred_val)

In [ ]:
xgb = XGBClassifier(random_state=seed)
xgb.fit(X_train_s, y_train)
y_pred_test = xgb.predict(X_test_s)
y_pred_val = xgb.predict(X_val_s)

resultados(modelo = 'XGBoost', y_pred_test=y_pred_test, y_test=y_test,
           y_val = y_val, y_pred_val = y_pred_val)

##Modelo de Árvore

In [ ]:
dt = DecisionTreeClassifier(random_state=seed)
dt.fit(X_train_s, y_train)
y_pred_test = dt.predict(X_test)
y_pred_val = dt.predict(X_val_s)

resultados(modelo = 'Decision Tree', y_pred_test=y_pred_test, y_test=y_test,
           y_val = y_val, y_pred_val = y_pred_val)

##Modelo de Vizinhos

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
y_pred_test = knn.predict(X_test)
y_pred_val = knn.predict(X_val_s)

resultados(modelo = 'KNN', y_pred_test=y_pred_test, y_test=y_test,
           y_val = y_val, y_pred_val = y_pred_val)

## Rede Neural MPL

In [ ]:
RN = keras.Sequential([
    layers.Input(shape=(16,)),

    layers.Dense(35, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(20, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(14, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(7, activation='softmax')
])

RN.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
y_train_np = np.array(y_train)
y_val_np = np.array(y_val)

history = RN.fit(
    X_train_s, y_train_np,
    validation_data=(X_val_s, y_val_np),
    epochs= 50,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='treino')
plt.plot(history.history['val_loss'], label='validação')
plt.title('Perda')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='treino')
plt.plot(history.history['val_accuracy'], label='validação')
plt.title('Acurácia')
plt.legend()
plt.show()

# Avaliando o Random Forest

In [ ]:
numerical_estm = df_train.drop(
    columns=['Crystal System', 'formula', 'composition'],
    errors='ignore'
).select_dtypes(include=np.number)

matrix = numerical_estm.corr(method='spearman')

# Aumenta o tamanho da figura
plt.figure(figsize=(9, 7))

plt.imshow(matrix, cmap='Blues')
plt.colorbar()
plt.title('Spearman Correlation')

# Ticks corretamente posicionados
plt.xticks(
    ticks=np.arange(len(matrix.columns)),
    labels=matrix.columns,
    rotation=45,
    ha='right',      # alinhar à direita evita sobreposição
    fontsize=9
)

plt.yticks(
    ticks=np.arange(len(matrix.columns)),
    labels=matrix.columns,
    fontsize=9
)

# Ajusta automaticamente margens
plt.tight_layout()

plt.show()

In [ ]:
# importância das features
importances = rf.feature_importances_

# dataframe organizado
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(
    feat_imp['feature'],
    feat_imp['importance']
)
plt.xlabel('Importância')
plt.title('')
plt.show()